# Smart MCQ Solver

## Library Imports

In [ ]:
import os
import string
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

input_dir = '/kaggle/input'
if os.path.exists(input_dir):
    for root, _, files in os.walk(input_dir):
        for file in files:
            print(os.path.join(root, file))


## Dummy Submission

In [ ]:
sample_sub = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
sample_sub.to_csv('submission.csv', index=False)


# Milestones

## Milestone 1

### Q1

In [ ]:
mcq_train_data = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
label_column_name = 'answer' if 'answer' in mcq_train_data.columns else 'label'
label_distribution = mcq_train_data[label_column_name].value_counts()
print(f"Q1 Answer: {label_distribution.max() + label_distribution.min()}")


### Q2

In [ ]:
def normalize_text_data(raw_text):
    clean_str = str(raw_text).lower()
    clean_str = clean_str.translate(str.maketrans('', '', string.punctuation))
    return clean_str

processed_prompts = mcq_train_data['prompt'].apply(normalize_text_data)
unique_vocabulary = set()
for prompt_text in processed_prompts:
    unique_vocabulary.update(prompt_text.split())
print(f"Q2 Answer: {len(unique_vocabulary)}")


### Q3

In [ ]:
first_prompt_tokens = processed_prompts.iloc[0].split()
filtered_tokens_no_stops = [t for t in first_prompt_tokens if t not in ENGLISH_STOP_WORDS]
print(f"Q3 Answer: {len(filtered_tokens_no_stops)}")


### Q4

In [ ]:
mcq_options = ['A', 'B', 'C', 'D', 'E']
text_corpora = []
for idx, data_row in mcq_train_data.iterrows():
    fused_text = str(data_row['prompt']) + ' ' + ' '.join([str(data_row[opt]) for opt in mcq_options])
    text_corpora.append(fused_text)
tfidf_vec = TfidfVectorizer(stop_words='english')
tfidf_vec.fit(text_corpora)
print(f"Q4 Answer: {len(tfidf_vec.get_feature_names_out())}")


### Q5

In [ ]:
prompt_vector = tfidf_vec.transform([str(mcq_train_data.iloc[0]['prompt'])])
option_a_vector = tfidf_vec.transform([str(mcq_train_data.iloc[0]['A'])])
similarity_score = cosine_similarity(prompt_vector, option_a_vector)[0][0]
print(f"Q5 Answer: {similarity_score:.4f}")

def calculate_map3(predictions_list, target_labels):
    ranking_scores = []
    for predictions, target in zip(predictions_list, target_labels):
        score = 0.0
        for rank_idx, pred in enumerate(predictions[:3]):
            if pred == target:
                score = 1.0 / (rank_idx + 1)
                break
        ranking_scores.append(score)
    return np.mean(ranking_scores)


### Q6

In [ ]:
tfidf_predictions = []
exact_first_choice_hits = 0
for _, data_row in mcq_train_data.iterrows():
    prompt_tfidf_vec = tfidf_vec.transform([str(data_row['prompt'])])
    similarity_map = {}
    for opt in mcq_options:
        opt_tfidf_vec = tfidf_vec.transform([str(data_row[opt])])
        similarity_map[opt] = cosine_similarity(prompt_tfidf_vec, opt_tfidf_vec)[0][0]
    sorted_options = sorted(similarity_map, key=similarity_map.get, reverse=True)
    tfidf_predictions.append(sorted_options)
    if sorted_options[0] == data_row[label_column_name]:
        exact_first_choice_hits += 1
accuracy_percentage = (exact_first_choice_hits / len(mcq_train_data)) * 100
print(f"Q6 Answer: {accuracy_percentage:.2f}%")


### Q7

In [ ]:
def compute_single_map3(prediction_list, target_label):
    for rank_idx, pred in enumerate(prediction_list[:3]):
        if pred == target_label:
            return 1.0 / (rank_idx + 1)
    return 0.0

print(f"Question 7 Verification Output: {compute_single_map3(['C', 'A', 'B'], 'C')}")


### Q8

In [ ]:
print(f"Question 8 Verification Output: {compute_single_map3(['D', 'B', 'E'], 'B')}")


### Q9

In [ ]:
top_three_labels = list(label_distribution.index[:3])
majority_baseline_preds = [top_three_labels] * len(mcq_train_data)
majority_class_map3 = calculate_map3(majority_baseline_preds, mcq_train_data[label_column_name])
print(f"Q9 Answer: {majority_class_map3:.4f}")


### Q10

In [ ]:
tfidf_pipeline_map3 = calculate_map3(tfidf_predictions, mcq_train_data[label_column_name])
print(f"Q10 Answer: {tfidf_pipeline_map3:.4f}")


## Milestone 2

### Setup — Imports, Seeds & Dataset

In [ ]:
import math
import os
import random
from pathlib import Path

# ── CPU lock ──────────────────────────────────────────────────────────────
# Hide all CUDA devices before torch/transformers initialise so the kernel-
# image mismatch error (AcceleratorError: no kernel image for this device)
# can never surface.  Every model and pipeline will automatically fall back
# to CPU.  Outputs are mathematically identical to GPU runs.
os.environ["CUDA_VISIBLE_DEVICES"] = ""
COMPUTE_DEVICE = "cpu"
# ─────────────────────────────────────────────────────────────────────────

import numpy as np
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoModel, AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)


def locate_train_file():
    priority_paths = [
        Path("/kaggle/input/smart-mcq-solver-challenge/train.csv"),
        Path("/kaggle/working/train.csv"),
        Path("data/train.csv"),
        Path("train.csv"),
    ]
    for p in priority_paths:
        if p.exists():
            return str(p)
    if Path("/kaggle/input").exists():
        kaggle_hits = sorted(Path("/kaggle/input").glob("**/train.csv"))
        if kaggle_hits:
            return str(kaggle_hits[0])
    raise FileNotFoundError("train.csv not found. Attach the competition dataset before running.")


train_file_path = locate_train_file()
m2_train_dataset = load_dataset("csv", data_files={"train": train_file_path})["train"]
m2_option_labels = ["A", "B", "C", "D", "E"]


### Q1

In [ ]:
def build_prompt_option_a(row):
    return {"combined_text": f"{row['prompt']} {row['A']}"}

dataset_with_combined = m2_train_dataset.map(build_prompt_option_a)
q1_combined_char_count = len(dataset_with_combined[51]["combined_text"])
print(q1_combined_char_count)


### Q2

In [ ]:
bert_tok = AutoTokenizer.from_pretrained("bert-base-uncased")
q2_vocab_size = bert_tok.vocab_size
print(q2_vocab_size)


### Q3

In [ ]:
q3_sep_token_id = bert_tok.sep_token_id
print(q3_sep_token_id)


### Q4

In [ ]:
prompt_text_list = list(m2_train_dataset["prompt"])
batch_encoding = bert_tok(
    prompt_text_list,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)
batch_encoding = {k: v.to(COMPUTE_DEVICE) for k, v in batch_encoding.items()}
q4_input_ids_shape = tuple(batch_encoding["input_ids"].shape)
print(q4_input_ids_shape)


### Q5

In [ ]:
bert_hidden_dim = 768
bert_num_heads = 12
q5_head_dim = bert_hidden_dim // bert_num_heads
print(q5_head_dim)


### Q6

In [ ]:
bert_encoder = AutoModel.from_pretrained("bert-base-uncased").to(COMPUTE_DEVICE)
row0_inputs = bert_tok(m2_train_dataset[0]["prompt"], return_tensors="pt")
row0_inputs = {k: v.to(COMPUTE_DEVICE) for k, v in row0_inputs.items()}
with torch.no_grad():
    row0_bert_out = bert_encoder(**row0_inputs)
q6_last_hidden_shape = tuple(row0_bert_out.last_hidden_state.shape)
print(q6_last_hidden_shape)


### Q7

In [ ]:
cls_vec = row0_bert_out.last_hidden_state[0, 0]
q7_cls_first5_sum = round(float(cls_vec[:5].sum().item()), 4)
print(q7_cls_first5_sum)


### Q8

In [ ]:
bert_attn_model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True,
).to(COMPUTE_DEVICE)
probe_text = "Light-ion fusion is a technique."
probe_inputs = bert_tok(probe_text, return_tensors="pt")
probe_inputs = {k: v.to(COMPUTE_DEVICE) for k, v in probe_inputs.items()}
with torch.no_grad():
    probe_out = bert_attn_model(**probe_inputs)
probe_tokens = bert_tok.convert_ids_to_tokens(probe_inputs["input_ids"][0])
fusion_idx = probe_tokens.index("fusion")
last_layer_head0 = probe_out.attentions[-1][0, 0]
q8_cls_to_fusion = round(float(last_layer_head0[0, fusion_idx].item()), 4)
print(q8_cls_to_fusion)


### Q9

In [ ]:
minilm_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=COMPUTE_DEVICE,
)
row0_prompt_emb = minilm_model.encode(m2_train_dataset[0]["prompt"], convert_to_tensor=True)
row0_option_b_emb = minilm_model.encode(m2_train_dataset[0]["B"], convert_to_tensor=True)
q9_prompt_b_sim = round(float(util.cos_sim(row0_prompt_emb, row0_option_b_emb).item()), 4)
print(q9_prompt_b_sim)


### Q10

In [ ]:
def compute_map_at_3(ground_truth, ranked_preds):
    running_total = 0.0
    for true_label, pred_list in zip(ground_truth, ranked_preds):
        if true_label in pred_list[:3]:
            running_total += 1.0 / (pred_list[:3].index(true_label) + 1)
    return running_total / len(ground_truth)


def tfidf_rank_options(row):
    option_texts = [row[lbl] for lbl in m2_option_labels]
    local_vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2))
    tfidf_mat = local_vec.fit_transform([row["prompt"]] + option_texts)
    sims = cosine_similarity(tfidf_mat[0:1], tfidf_mat[1:]).ravel()
    ranked_idx = np.argsort(sims)[::-1]
    return [m2_option_labels[i] for i in ranked_idx[:3]]


def batch_encode(texts, encoder, chunk_size=64):
    return encoder.encode(
        texts, batch_size=chunk_size, convert_to_tensor=True,
        normalize_embeddings=True, show_progress_bar=False,
    )


tfidf_top3 = [tfidf_rank_options(row) for row in m2_train_dataset]
prompt_embeddings = batch_encode(list(m2_train_dataset["prompt"]), minilm_model)
option_embeddings = {lbl: batch_encode(list(m2_train_dataset[lbl]), minilm_model) for lbl in m2_option_labels}

minilm_top3 = []
for row_idx in range(len(m2_train_dataset)):
    scores = [
        float(util.cos_sim(prompt_embeddings[row_idx], option_embeddings[lbl][row_idx]).item())
        for lbl in m2_option_labels
    ]
    ranked_idx = np.argsort(scores)[::-1]
    minilm_top3.append([m2_option_labels[i] for i in ranked_idx[:3]])

ground_truth = list(m2_train_dataset["answer"])
q10_minilm_map3 = compute_map_at_3(ground_truth, minilm_top3)
q10_rescue_count = sum(
    true_lbl not in tfidf_pred and true_lbl in minilm_pred
    for true_lbl, tfidf_pred, minilm_pred in zip(ground_truth, tfidf_top3, minilm_top3)
)
print(round(q10_minilm_map3, 4))
print(q10_rescue_count)


### Q11

In [ ]:
zs_classifier = pipeline("zero-shot-classification", device=-1)
row1_candidates = [m2_train_dataset[1][lbl] for lbl in ["A", "B", "C"]]
row1_softmax_result = zs_classifier(
    m2_train_dataset[1]["prompt"],
    candidate_labels=row1_candidates,
)
q11_top_score = round(float(row1_softmax_result["scores"][0]), 4)
print(q11_top_score)


### Q12

In [ ]:
row1_sigmoid_result = zs_classifier(
    m2_train_dataset[1]["prompt"],
    candidate_labels=row1_candidates,
    multi_label=True,
)
softmax_sum = sum(row1_softmax_result["scores"])
sigmoid_sum = sum(row1_sigmoid_result["scores"])
q12_score_diff = abs(softmax_sum - sigmoid_sum)
print(q12_score_diff)


### Q13

In [ ]:
flan_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
flan_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to(COMPUTE_DEVICE)

binary_prompt = (
    f"Question: {m2_train_dataset[0]['prompt']}. "
    f"Is the correct answer A: {m2_train_dataset[0]['A']} "
    f"or B: {m2_train_dataset[0]['B']}? "
    "Answer with just the letter A or B."
)
input_ids = flan_tokenizer(binary_prompt, return_tensors="pt").input_ids.to(COMPUTE_DEVICE)
with torch.no_grad():
    outputs = flan_model.generate(input_ids, max_new_tokens=5)
q13_generated_answer = flan_tokenizer.decode(outputs[0], skip_special_tokens=True)
print(q13_generated_answer)


## Milestone 3

### Install — FAISS

In [ ]:
!pip install faiss-cpu --quiet


### Setup — Imports, Models & Knowledge Base

In [ ]:
import faiss
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# CPU lock inherited from M2 setup; re-assert here so M3 is self-contained
os.environ["CUDA_VISIBLE_DEVICES"] = ""

# ── Data ─────────────────────────────────────────────────────────────────
# locate_train_file() is defined in the M2 setup cell above
m3_train_file = locate_train_file()
m3_train = pd.read_csv(m3_train_file)
m3_option_letters = ["A", "B", "C", "D", "E"]

# ── Knowledge Base ────────────────────────────────────────────────────────
# Each entry = the correct answer text for that training row
m3_kb = [
    str(m3_row[m3_row["answer"]])
    for _, m3_row in m3_train.iterrows()
]

# ── Embedding model (CPU) ─────────────────────────────────────────────────
m3_embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cpu",
)
m3_kb_embeddings = m3_embedder.encode(
    m3_kb,
    show_progress_bar=False,
    convert_to_numpy=True,
).astype("float32")

# ── FAISS flat L2 index ───────────────────────────────────────────────────
m3_index = faiss.IndexFlatL2(m3_kb_embeddings.shape[1])
m3_index.add(m3_kb_embeddings)

# ── Zero-shot classifier (CPU) ────────────────────────────────────────────
m3_zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=-1,
)

# ── Cross-encoder re-ranker (CPU) ─────────────────────────────────────────
m3_cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cpu",
)

# ── BERT tokenizer (for token-count questions) ────────────────────────────
m3_bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")


### Q1

In [ ]:
m3_row_150 = m3_train.iloc[150]
m3_prompt_150 = str(m3_row_150["prompt"])
m3_labels_150 = [str(m3_row_150[m3_letter]) for m3_letter in m3_option_letters]
m3_true_letter_150 = str(m3_row_150["answer"])
m3_true_doc_150 = str(m3_row_150[m3_true_letter_150])

m3_q1_output = m3_zero_shot(m3_prompt_150, candidate_labels=m3_labels_150)
m3_q1_score_lookup = dict(zip(m3_q1_output["labels"], m3_q1_output["scores"]))
m3_q1_correct_probability = round(float(m3_q1_score_lookup[m3_true_doc_150]), 3)
print("Q1:", m3_q1_correct_probability)


### Q2

In [ ]:
m3_prompt_150_embedding = m3_embedder.encode(
    [m3_prompt_150], show_progress_bar=False, convert_to_numpy=True,
).astype("float32")

m3_q2_distances, m3_q2_indices = m3_index.search(m3_prompt_150_embedding, 10)
m3_q2_retrieved_indices = m3_q2_indices[0].tolist()

m3_q2_true_rank = (
    m3_q2_retrieved_indices.index(150) + 1
    if 150 in m3_q2_retrieved_indices
    else None
)
print("Q2:", m3_q2_true_rank)


### Q3

In [ ]:
m3_q3_docs = [m3_kb[m3_doc_idx] for m3_doc_idx in m3_q2_retrieved_indices]
m3_q3_pairs = [(m3_prompt_150, m3_doc) for m3_doc in m3_q3_docs]
m3_q3_scores = m3_cross_encoder.predict(m3_q3_pairs)

m3_q3_ranked_items = sorted(
    zip(m3_q2_retrieved_indices, m3_q3_docs, m3_q3_scores),
    key=lambda m3_item: float(m3_item[2]),
    reverse=True,
)
m3_q3_ranked_indices = [m3_item[0] for m3_item in m3_q3_ranked_items]
m3_q3_true_rank = (
    m3_q3_ranked_indices.index(150) + 1
    if 150 in m3_q3_ranked_indices
    else None
)
print("Q3:", m3_q3_true_rank)


### Q4

In [ ]:
m3_row_42 = m3_train.iloc[42]
m3_prompt_42 = str(m3_row_42["prompt"])
m3_prompt_42_embedding = m3_embedder.encode(
    [m3_prompt_42], show_progress_bar=False, convert_to_numpy=True,
).astype("float32")

m3_q4_distances, m3_q4_indices = m3_index.search(m3_prompt_42_embedding, 5)
m3_q4_docs = [m3_kb[m3_doc_idx] for m3_doc_idx in m3_q4_indices[0].tolist()]
m3_q4_context = " ".join(m3_q4_docs)
m3_q4_rag_text = f"Context: {m3_q4_context} Question: {m3_prompt_42}"

m3_q4_tokens = m3_bert_tokenizer(
    m3_q4_rag_text,
    truncation=False,
    add_special_tokens=True,
)
m3_q4_total_tokens = len(m3_q4_tokens["input_ids"])
print("Q4:", m3_q4_total_tokens)


### Q5

In [ ]:
m3_q5_rag_text = f"Context: {m3_true_doc_150} Question: {m3_prompt_150}"
m3_q5_output = m3_zero_shot(m3_q5_rag_text, candidate_labels=m3_labels_150)
m3_q5_score_lookup = dict(zip(m3_q5_output["labels"], m3_q5_output["scores"]))
m3_q5_correct_probability = round(float(m3_q5_score_lookup[m3_true_doc_150]), 3)
print("Q5:", m3_q5_correct_probability)


### Q6

In [ ]:
m3_wrong_doc_999 = str(m3_kb[999])
m3_q6_rag_text = f"Context: {m3_wrong_doc_999} Question: {m3_prompt_150}"
m3_q6_output = m3_zero_shot(m3_q6_rag_text, candidate_labels=m3_labels_150)
m3_q6_score_lookup = dict(zip(m3_q6_output["labels"], m3_q6_output["scores"]))
m3_q6_correct_probability = round(float(m3_q6_score_lookup[m3_true_doc_150]), 3)
print("Q6:", m3_q6_correct_probability)


### Q7

In [ ]:
m3_q7_hits = 0
for m3_eval_idx in range(100):
    m3_eval_row = m3_train.iloc[m3_eval_idx]
    m3_eval_prompt = str(m3_eval_row["prompt"])
    m3_eval_correct_doc = str(m3_eval_row[str(m3_eval_row["answer"])])
    m3_eval_emb = m3_embedder.encode(
        [m3_eval_prompt], show_progress_bar=False, convert_to_numpy=True,
    ).astype("float32")
    _, m3_eval_hit_indices = m3_index.search(m3_eval_emb, 5)
    m3_eval_docs = [m3_kb[i] for i in m3_eval_hit_indices[0].tolist()]
    if any(m3_eval_correct_doc in m3_doc for m3_doc in m3_eval_docs):
        m3_q7_hits += 1
m3_q7_hit_rate = round((m3_q7_hits / 100) * 100, 1)
print("Q7:", m3_q7_hit_rate)


### Q8

In [ ]:
def m3_map_at_3(m3_ranked_letters, m3_actual_letter):
    if m3_actual_letter not in m3_ranked_letters[:3]:
        return 0.0
    return 1.0 / (m3_ranked_letters[:3].index(m3_actual_letter) + 1)


m3_q8_scores = []
for m3_eval_idx in range(20):
    m3_eval_row = m3_train.iloc[m3_eval_idx]
    m3_eval_prompt = str(m3_eval_row["prompt"])
    m3_eval_labels = [str(m3_eval_row[m3_letter]) for m3_letter in m3_option_letters]
    m3_eval_answer_letter = str(m3_eval_row["answer"])

    m3_eval_emb = m3_embedder.encode(
        [m3_eval_prompt], show_progress_bar=False, convert_to_numpy=True,
    ).astype("float32")
    _, m3_eval_hit_indices = m3_index.search(m3_eval_emb, 5)
    m3_eval_docs = [m3_kb[i] for i in m3_eval_hit_indices[0].tolist()]

    m3_eval_pairs = [(m3_eval_prompt, m3_doc) for m3_doc in m3_eval_docs]
    m3_eval_cross_scores = m3_cross_encoder.predict(m3_eval_pairs)

    m3_best_doc = m3_eval_docs[int(np.argmax(m3_eval_cross_scores))]
    m3_eval_rag_text = f"Context: {m3_best_doc} Question: {m3_eval_prompt}"
    m3_eval_output = m3_zero_shot(m3_eval_rag_text, candidate_labels=m3_eval_labels)

    m3_label_to_letter = {
        str(m3_eval_row[m3_letter]): m3_letter
        for m3_letter in m3_option_letters
    }
    m3_ranked_letters = [
        m3_label_to_letter[m3_label]
        for m3_label in m3_eval_output["labels"]
    ]
    m3_q8_scores.append(m3_map_at_3(m3_ranked_letters, m3_eval_answer_letter))

m3_q8_average_map3 = round(float(np.mean(m3_q8_scores)), 3)
print("Q8:", m3_q8_average_map3)
